# Campus ML: учебная лаборатория Titanic

Этот авторский ноутбук сопровождает маршрут SUAI Hack Club. Это рабочая основа, которую нужно дополнить собственными графиками, экспериментами и выводами.

## До начала
1. Скачай **train.csv** и **test.csv** с `/challenge` действующего сайта клуба и положи рядом с ноутбуком. Это исторические данные Titanic из OpenML (ID 40945), разбиение клуба: 1 009 train и 300 test. Файлы Kaggle сюда не подходят. Не подставляй готовые исторические ответы: организатор проверяет ход решения.
2. В отдельном окружении установи `pandas scikit-learn matplotlib jupyter` (Python 3.11+). Открой Jupyter из этого окружения.
3. Выполняй ячейки по порядку. Не публикуй личные ссылки для входа, секреты и закрытые ответы.
4. Время маршрута включает чтение, собственные эксперименты и анализ, а не только запуск готовых ячеек.

**Правило проверки:** используем 75% train как development, 25% как holdout. Подбор проходит только внутри development. Holdout открываем после фиксации модели. Если меняешь подход по результатам holdout, он больше не является независимой оценкой.

Материалы: [pandas](https://pandas.pydata.org/docs/getting_started/intro_tutorials/), [scikit-learn: CV](https://scikit-learn.org/stable/modules/cross_validation.html), [Inria MOOC](https://inria.github.io/scikit-learn-mooc/).
Источник: [OpenML Titanic 40945](https://www.openml.org/d/40945), Frank E. Harrell Jr., Thomas Cason. Поля boat и body исключены как утечка. Семьи могут создавать зависимость между строками: обычное случайное разбиение не проверяет перенос на новые семьи.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)
assert Path("train.csv").exists() and Path("test.csv").exists(), "Положи CSV рядом с ноутбуком"

## 1. Схема и целостность
Пока проверяем только структуру. Исследование зависимостей от survived делаем на development.

**Твоя работа:** опиши смысл строки, единицы каждого поля, момент прогноза и возможные ограничения данных. Не включай id в признаки.

In [ ]:
train = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")
numeric = ["pclass", "age", "sibsp", "parch", "fare"]
categorical = ["sex", "embarked"]
features = numeric + categorical
assert set(train.columns) == set(["id", "survived"] + features)
assert set(test_data.columns) == set(["id"] + features)
assert train["id"].is_unique and test_data["id"].is_unique
assert set(train["id"]).isdisjoint(test_data["id"])
assert train["survived"].isin([0, 1]).all()
print("Размеры:", train.shape, test_data.shape)
print(train.dtypes)
print("Пропуски:\n", train.isna().sum())

development, holdout = train_test_split(train, test_size=0.25, stratify=train["survived"], random_state=42)
assert set(development["id"]).isdisjoint(holdout["id"])
X_dev, y_dev = development[features], development["survived"]
X_hold, y_hold = holdout[features], holdout["survived"]
pd.DataFrame({"id": train["id"], "part": np.where(train.index.isin(development.index), "development", "holdout")}).to_csv("split.csv", index=False)
print("Разбиение:", len(development), len(holdout))

## 2. Исследование development
Ниже один пример. Добавь **ещё два разных графика**, подписи осей и выводы. Для каждой группы указывай число строк. Связь признаков с survived не доказывает причинность.

Запиши три гипотезы до обучения: вопрос → наблюдение → альтернативное объяснение → проверка.

In [ ]:
print("Доли классов на development:\n", y_dev.value_counts(normalize=True))
by_class = development.groupby("pclass")["survived"].agg(["mean", "size"])
print(by_class)
by_class["mean"].plot.bar(title="Доля выживших по классу билета", ylabel="Доля survived=1", xlabel="Класс билета")
plt.tight_layout()
plt.show()

### Мои наблюдения
- Наблюдение 1 и основание:
- Наблюдение 2 и основание:
- Наблюдение 3 и основание:
- Гипотезы для экспериментов:

## 3. Pipeline
Числовые пропуски заполняются медианой, затем выполняется масштабирование. Категории sex и embarked заполняются наиболее частым значением и кодируются one-hot. **Каждый fit происходит внутри обучающего фолда.**

Объясни, почему pipeline нужно передавать в CV целиком. Масштабирование оставлено общим для сравнения; лесу оно само по себе не требуется.

In [ ]:
def make_model(estimator):
    preprocessing = ColumnTransformer([
        ("numbers", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
        ("category", Pipeline([("fill", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])
    return Pipeline([("prepare", preprocessing), ("model", estimator)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
candidates = {
    "dummy_majority": make_model(DummyClassifier(strategy="most_frequent")),
    "dummy_positive": make_model(DummyClassifier(strategy="constant", constant=1)),
    "logistic": make_model(LogisticRegression(max_iter=1000, random_state=42)),
    "forest": make_model(RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=5, random_state=42, n_jobs=1)),
}
rows = []
for name, model in candidates.items():
    result = cross_validate(model, X_dev, y_dev, cv=cv, scoring="f1", return_train_score=True)
    rows.append({"name": name, "cv_f1": result["test_score"].mean(), "cv_sd": result["test_score"].std(),
                 "train_f1": result["train_score"].mean(), "fit_seconds": result["fit_time"].mean()})
    print(name, "F1 по фолдам:", result["test_score"])
results = pd.DataFrame(rows).set_index("name")
print(results.sort_values("cv_f1", ascending=False))

## 4. Маленький подбор параметров
Четыре комбинации × пять фолдов + финальный refit. Это учебный бюджет, не соревнование вычислительных мощностей.

**Твоя работа:** добавь неглубокое дерево и один осмысленный эксперимент с признаками или параметрами, чтобы получить минимум шесть запусков. Запиши гипотезы, параметры и результаты. Не меняй holdout.

Если добавляешь признаки, одинаковое преобразование должно применяться в development, holdout и test. Для начала оставь схему неизменной и сравни настройки.

In [ ]:
search = GridSearchCV(
    candidates["forest"],
    {"model__max_depth": [3, 6], "model__min_samples_leaf": [3, 8]},
    scoring="f1", cv=cv, n_jobs=1, return_train_score=True,
)
search.fit(X_dev, y_dev)
print("Лучшие параметры:", search.best_params_)
print("Лучший CV F1:", search.best_score_)
print(pd.DataFrame(search.cv_results_)[["params", "mean_test_score", "std_test_score", "mean_train_score"]])

best_baseline_name = results["cv_f1"].idxmax()
if search.best_score_ > results.loc[best_baseline_name, "cv_f1"]:
    chosen = clone(search.best_estimator_)
    choice_name = "tuned_forest"
else:
    chosen = clone(candidates[best_baseline_name])
    choice_name = best_baseline_name
print("Учебный выбор по development:", choice_name)
print("Параметры:", chosen.get_params()["model"])

### Зафиксируй решение ДО следующей ячейки
- Мой вариант и обоснование:
- Признаки и обработка:
- Параметры и seed:
- Порог: в этом ноутбуке используется стандартный predict; для вероятностных классификаторов здесь это порог 0,5.
- Коммит и версии библиотек:
- Что я ожидаю от holdout:

**Не открывай holdout ради выбора между кандидатами.** Автоматический выбор выше служит примером; сначала заверши собственные эксперименты на development.

## 5. Финальная локальная оценка
После фиксации обучаем chosen на development и один раз оцениваем holdout. Повторный запуск для воспроизведения допустим; подгонка по результату лишает оценку независимости.

In [ ]:
chosen.fit(X_dev, y_dev)
prediction = chosen.predict(X_hold)
print({
    "F1": f1_score(y_hold, prediction, zero_division=0),
    "precision": precision_score(y_hold, prediction, zero_division=0),
    "recall": recall_score(y_hold, prediction, zero_division=0),
})
ConfusionMatrixDisplay.from_predictions(y_hold, prediction, labels=[0, 1])
plt.title("Фиксированная модель на holdout")
plt.tight_layout()
plt.show()
errors = holdout.copy()
errors["prediction"] = prediction
errors = errors[errors["survived"] != errors["prediction"]]
print("Ошибок:", len(errors))
print(errors.head(10).to_string(index=False))

## 6. Разбор ошибок и важности
Выбери минимум пять ошибок (или все, если их меньше). Для каждой укажи FP/FN, возможное объяснение и какую информацию хотелось бы получить.

Permutation importance ниже — описание уже зафиксированной модели. Если используешь его для нового выбора признаков, нужен отдельный независимый набор оценки. Коррелированные признаки могут скрывать важность друг друга; важность не означает причинность.

In [ ]:
importance = permutation_importance(chosen, X_hold, y_hold, scoring="f1", n_repeats=5, random_state=42, n_jobs=1)
importance_table = pd.DataFrame({"feature": features, "mean": importance.importances_mean, "std": importance.importances_std})
print(importance_table.sort_values("mean", ascending=False))

### Мой отчёт об ограничениях
- Какие ошибки встречаются:
- На сколько примеров опирается каждый вывод:
- Что показывает и чего не доказывает важность:
- Какие решения изменены после просмотра holdout (если менялись):
- Что нужно проверить на новых данных:

## 7. Финальная модель и CSV
Зафиксированный подход можно переобучить на всех размеченных данных. Не называй оценку на этих же данных независимой. Сохраняй ID вместе с соответствующими предсказаниями.

In [ ]:
final_model = clone(chosen)
final_model.fit(train[features], train["survived"])
submission = pd.DataFrame({"id": test_data["id"], "survived": final_model.predict(test_data[features]).astype(int)})
assert list(submission.columns) == ["id", "survived"]
assert len(submission) == len(test_data) == 300
assert submission["id"].is_unique
assert set(submission["id"]) == set(test_data["id"])
assert submission["survived"].isin([0, 1]).all()
assert not submission.isna().any().any()
submission.to_csv("submission.csv", index=False)
check = pd.read_csv("submission.csv")
assert check.equals(submission)
print("submission.csv готов. Допиши анализ и README до отправки.")

## 8. Чек-лист передачи
- [ ] Есть паспорт данных и три графика с выводами.
- [ ] Есть минимум шесть осмысленных экспериментов и журнал гипотез.
- [ ] Описаны split, метрика, параметры и роль holdout.
- [ ] Разобраны ошибки и ограничения интерпретации.
- [ ] Ноутбук работает сверху вниз после перезапуска ядра.
- [ ] README содержит версии пакетов и команды запуска.
- [ ] Ссылка доступна организатору, в коде и выводах нет секретов.
- [ ] Указано, как использовался ИИ и что проверено человеком.

Отправь CSV и ссылку на решение через `/challenge`. Автоматический рейтинг измеряет качество предсказаний; содержание ноутбука проверяет организатор.

### Дополнительно
Прочитай [настройку порога](https://scikit-learn.org/stable/modules/classification_threshold.html). Сделай отдельный эксперимент только внутри development. Для новой честной финальной оценки после работы с текущим holdout потребуются новые данные.